In [ ]:

#making of dataset for lecturer upload format from Y1S1 to Y2S2
import pandas as pd


marks_gpa_path = r"C:\Users\User\Desktop\Final Year Project\22b6023 Muizzul\3 Processed Data\MARKS GPA.xlsx"
transcript_path = r"C:\Users\User\Desktop\Final Year Project\Data\Transcript.xlsx"
details_path = r"C:\Users\User\Desktop\Final Year Project\Data\ListOfStudPersDetails.xlsx"

output_path = r"C:\Users\User\Desktop\Final Year Project\22b6023 Muizzul\3 Processed Data\Lecturer_Upload_Format_Y1S1_to_Y2S2.xlsx"

target_sheets = ["Y1S1", "Y1S2", "Y2S1", "Y2S2"]
details_sheet = "850 student personal details"


details_df = pd.read_excel(details_path, sheet_name=details_sheet)
details_df.columns = details_df.columns.astype(str).str.strip()


possible_regno_cols = ["REGNO", "Reg No", "REG NO", "RegNo", "Registration No"]
possible_major_cols = ["MAJOR", "Major", "Programme", "Program", "Program Name", "Major name"]

regno_col = next((c for c in details_df.columns if c in possible_regno_cols), None)
major_col = next((c for c in details_df.columns if c in possible_major_cols), None)

if regno_col is None:
    raise ValueError(f"Could not find REGNO column in {details_sheet}. Found columns: {list(details_df.columns)}")

if major_col is None:
    raise ValueError(f"Could not find MAJOR column in {details_sheet}. Found columns: {list(details_df.columns)}")

details_df[regno_col] = details_df[regno_col].astype(str).str.strip()
details_df[major_col] = details_df[major_col].astype(str).str.strip()

major_map = (
    details_df[[regno_col, major_col]]
    .dropna(subset=[regno_col])
    .drop_duplicates(subset=[regno_col])
    .rename(columns={regno_col: "REGNO", major_col: "MAJOR"})
)


output_sheets = {}

for sh in target_sheets:
    print(f"Processing {sh}...")


    gpa_df = pd.read_excel(marks_gpa_path, sheet_name=sh)
    gpa_df.columns = gpa_df.columns.astype(str).str.strip()

    if "REGNO" not in gpa_df.columns:
        raise ValueError(f"{marks_gpa_path} sheet {sh} does not contain REGNO")
    if "GPA" not in gpa_df.columns:
        raise ValueError(f"{marks_gpa_path} sheet {sh} does not contain GPA")

    gpa_df["REGNO"] = gpa_df["REGNO"].astype(str).str.strip()
    gpa_df["GPA"] = pd.to_numeric(gpa_df["GPA"], errors="coerce")

    gpa_map = (
        gpa_df[["REGNO", "GPA"]]
        .dropna(subset=["REGNO"])
        .drop_duplicates(subset=["REGNO"])
    )


    trans_df = pd.read_excel(transcript_path, sheet_name=sh)
    trans_df.columns = trans_df.columns.astype(str).str.strip()

    required_transcript_cols = ["REGNO", "MODULE_CODE", "MC", "MARKS"]
    missing_cols = [c for c in required_transcript_cols if c not in trans_df.columns]
    if missing_cols:
        raise ValueError(f"{transcript_path} sheet {sh} missing columns: {missing_cols}")

    trans_df["REGNO"] = trans_df["REGNO"].astype(str).str.strip()
    trans_df["MODULE_CODE"] = trans_df["MODULE_CODE"].astype(str).str.strip()
    trans_df["MC"] = pd.to_numeric(trans_df["MC"], errors="coerce")
    trans_df["MARKS"] = pd.to_numeric(trans_df["MARKS"], errors="coerce")

    # Keep only useful rows
    trans_df = trans_df[trans_df["REGNO"].notna() & (trans_df["REGNO"] != "")]
    trans_df = trans_df[trans_df["MODULE_CODE"].notna() & (trans_df["MODULE_CODE"] != "nan")]

    #  Merge GPA + MAJOR 
    merged_df = trans_df.merge(gpa_map, on="REGNO", how="left")
    merged_df = merged_df.merge(major_map, on="REGNO", how="left")

    # Final output columns 
    final_df = merged_df[["REGNO", "MAJOR", "MODULE_CODE", "MC", "MARKS", "GPA"]].copy()


    final_df = final_df.sort_values(by=["REGNO", "MODULE_CODE"]).reset_index(drop=True)

    output_sheets[sh] = final_df
    print(f"{sh}: {len(final_df)} rows")


with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sh, df_out in output_sheets.items():
        df_out.to_excel(writer, sheet_name=sh, index=False)

print("\nSaved lecturer upload workbook to:")
print(output_path)

Processing Y1S1...
Y1S1: 2976 rows
Processing Y1S2...
Y1S2: 4085 rows
Processing Y2S1...
Y2S1: 4067 rows
Processing Y2S2...
Y2S2: 4108 rows

Saved lecturer upload workbook to:
C:\Users\User\Desktop\Final Year Project\22b6023 Muizzul\3 Processed Data\Lecturer_Upload_Format_Y1S1_to_Y2S2.xlsx
